# Disease Prediction from Medical Data
### CodeAlpha Machine Learning Internship
**Name:** Sumit Kumar Mahto &nbsp;|&nbsp; **ID:** CA/DF1/47253

Here I'm trying to predict whether a patient has diabetes based on diagnostic measurements like glucose level, BMI and age. I'm using the Pima Indians Diabetes dataset which is a well-known benchmark in medical ML.

I'll compare 4 algorithms — Logistic Regression, SVM, Random Forest and XGBoost.


In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report)
import joblib

print("all libraries loaded!")

## Step 1 — Load the Dataset

I'm loading the Pima Indians Diabetes dataset directly from a public URL. If that doesn't work it falls back to synthetic data.

In [ ]:
url  = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
        "Insulin", "BMI", "DiabetesPedigree", "Age", "Outcome"]
try:
    df = pd.read_csv(url, names=cols)
    print("loaded from URL!")
except:
    np.random.seed(42)
    n = 768
    df = pd.DataFrame({
        "Pregnancies":      np.random.randint(0, 17, n),
        "Glucose":          np.random.randint(50, 200, n),
        "BloodPressure":    np.random.randint(30, 122, n),
        "SkinThickness":    np.random.randint(0, 99, n),
        "Insulin":          np.random.randint(0, 846, n),
        "BMI":              np.round(np.random.uniform(0, 67, n), 1),
        "DiabetesPedigree": np.round(np.random.uniform(0.07, 2.42, n), 3),
        "Age":              np.random.randint(21, 81, n),
        "Outcome":          np.random.choice([0, 1], n, p=[0.65, 0.35])
    })
    print("using synthetic data (URL failed)")

print(f"shape: {df.shape}")
df.head()

In [ ]:
print("class distribution:")
print(df["Outcome"].value_counts())
print(f"
diabetes rate: {df['Outcome'].mean():.1%}")

In [ ]:
# correlation heatmap — useful to see which features relate to the outcome
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5)
plt.title("feature correlations", fontsize=12)
plt.tight_layout(); plt.show()

## Step 2 — Handle Missing Values

Some features have 0 values that are biologically impossible — e.g. a Glucose of 0. These are actually missing values in disguise.

In [ ]:
# zeros in these columns mean missing data
cols_with_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols_with_zeros] = df[cols_with_zeros].replace(0, np.nan)

print("missing values after replacing impossible zeros:")
print(df.isnull().sum())

In [ ]:
# distributions split by outcome
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), df.columns[:-1]):
    for outcome, color, label in [(0, "#4CAF50", "healthy"), (1, "#FF5722", "diabetic")]:
        df[df["Outcome"] == outcome][col].dropna().plot(
            kind="hist", alpha=0.5, ax=ax, bins=20, color=color, label=label
        )
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions by Outcome", fontsize=12)
plt.tight_layout(); plt.show()

## Step 3 — Split & Preprocess

In [ ]:
features = [c for c in df.columns if c != "Outcome"]
X = df[features]
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train : {X_train.shape[0]} samples")
print(f"test  : {X_test.shape[0]} samples")
print(f"diabetes rate in train: {y_train.mean():.1%}")

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # fill missing with median
    ("scaler",  StandardScaler())
])

## Step 4 — Train & Cross-Validate

Comparing 4 algorithms — logistic regression as a baseline, then SVM, Random Forest and XGBoost.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "SVM": Pipeline([
        ("pre", preprocessor),
        ("clf", SVC(probability=True, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("pre", preprocessor),
        ("clf", XGBClassifier(n_estimators=100, random_state=42,
                               use_label_encoder=False, eval_metric="logloss"))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("5-fold cross-validation ROC-AUC:")
print("-" * 38)
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:<22} {scores.mean():.4f} ± {scores.std():.4f}")

## Step 5 — Test Set Results

In [ ]:
results = []
trained = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    trained[name] = pipeline

    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results).set_index("Model")
results_df

## Step 6 — Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Disease Prediction — Results", fontsize=13, fontweight="bold")

# roc curves
ax = axes[0]
colors = ["#2196F3", "#9C27B0", "#4CAF50", "#FF5722"]
for (name, pipeline), color in zip(trained.items(), colors):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} ({auc:.2f})", color=color, lw=2)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("ROC curves"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(fontsize=7); ax.grid(alpha=0.3)

# confusion matrix
ax = axes[1]
best_name = results_df["ROC-AUC"].idxmax()
cm = confusion_matrix(y_test, trained[best_name].predict(X_test))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", ax=ax,
            xticklabels=["healthy", "diabetic"],
            yticklabels=["healthy", "diabetic"])
ax.set_title(f"confusion matrix
({best_name})")
ax.set_ylabel("actual"); ax.set_xlabel("predicted")

# xgboost feature importance
ax = axes[2]
xgb_clf  = trained["XGBoost"].named_steps["clf"]
feat_imp = pd.Series(xgb_clf.feature_importances_, index=features).sort_values()
feat_imp.plot(kind="barh", ax=ax, color="#FF5722", alpha=0.85)
ax.set_title("feature importances
(XGBoost)")
ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("task4_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved: task4_results.png")

## Step 7 — Save & Predict

In [ ]:
joblib.dump(trained[best_name], "disease_model.pkl")
print(f"saved: disease_model.pkl  (best: {best_name})")

# test on a sample patient
patient = pd.DataFrame([{
    "Pregnancies": 3, "Glucose": 148, "BloodPressure": 72,
    "SkinThickness": 35, "Insulin": 0, "BMI": 33.6,
    "DiabetesPedigree": 0.627, "Age": 50
}])

model = joblib.load("disease_model.pkl")
prob  = model.predict_proba(patient)[0][1]
label = "DIABETIC RISK ⚠️" if prob >= 0.5 else "LOW RISK ✅"

print(f"
patient: age 50, glucose 148, bmi 33.6, pregnancies 3")
print(f"diabetes probability: {prob:.2%}")
print(f"result: {label}")